# 06 — Entropic Quantum OT and the Gibbs Kernel

Smooth the coupling SDP the way module 03 smoothed the Kantorovich LP — add a von Neumann entropy bonus $-\varepsilon\,S(\rho_{AB})$ — and watch the **matrix-exponential Gibbs kernel** $K = \exp(-C/\varepsilon)$ appear at the centre of the regularised problem.

**Prerequisites:** `04/05_solving_qot_in_cvxpy`, `03/09_entropic_regularization`.

**What you'll be able to do**
- Write the **entropic quantum OT** objective and say why the entropy bonus makes it strongly convex with a single, smooth optimum.
- Build the **matrix-exponential Gibbs kernel** $K = \exp(-C/\varepsilon)$ and recognise it as the operator lift of the entrywise classical kernel $K_{ij} = e^{-C_{ij}/\varepsilon}$ from `03/09`.
- Solve the **entropic QOT SDP** in cvxpy with one extra term, and read its objective as transport cost minus $\varepsilon$ times entropy.
- See the entropic plan come out **smoother** — higher entropy, more spread — than the sharp coupling the unregularised SDP returned in `04/05`.

In `04/05` you pointed the coupling SDP at $|+\rangle\langle+|$ versus $I/2$ and read the single optimal coupling it returned — the sharp, sometimes brittle minimiser. Module 03 met exactly this brittleness in the classical world and answered it with a small, fertile idea: reward the plan a little for being spread out. That entropic bonus made the LP smooth and strongly convex, and it put one object at the centre of everything — the Gibbs kernel. Here we lift that idea to operators. The entropy becomes von Neumann's, the kernel becomes a matrix exponential, and the rest of the story rhymes line for line with `03/09`. The explicit *algorithm* that walks to this optimum is the next notebook; here we meet the regularised problem itself and the kernel that anchors it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.composite import partial_trace, tensor
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_PLUS
from qot_course.quantum_ot.sdp import quadratic_position_cost, quantum_ot_sdp
from qot_course.quantum_ot.sinkhorn import (
    matrix_gibbs_kernel,
    quantum_sinkhorn_sdp,
    quantum_sinkhorn_cost,
)

np.random.seed(42)
viz.use_course_style()

## The entropic quantum OT problem

The coupling SDP of `04/05` minimised a linear objective, $\mathrm{tr}(C\,\rho_{AB})$, over the set of quantum couplings $\Pi(\rho_A, \rho_B)$. A linear objective on a convex set can have a flat optimal face — a whole ridge of minimisers tied for the best cost — which is why the returned plan can be sharp and can jump as the inputs move. Module 03 fixed this for the Kantorovich LP by adding an entropy bonus (Cuturi, 2013). We add the operator version of the very same bonus:

$$
\boxed{\;\rho^\star_{AB,\,\varepsilon} \,=\, \arg\min_{\rho_{AB}\,\in\,\Pi(\rho_A, \rho_B)}\;
       \mathrm{tr}(C\,\rho_{AB}) \;-\; \varepsilon\, S(\rho_{AB})\;}
$$

where $S(\rho_{AB}) = -\mathrm{tr}(\rho_{AB}\log\rho_{AB})$ is the **von Neumann entropy** — the spectral cousin of the Shannon entropy $H(P) = -\sum_{ij} P_{ij}\log P_{ij}$ that `03/09` used. The same three consequences follow, now lifted from probability vectors to density matrices:

- **Strong convexity.** Von Neumann entropy is strictly concave on the cone of density matrices, so $-\varepsilon S$ is strictly convex. The minimiser becomes **unique** and varies **smoothly** with $\rho_A$, $\rho_B$ and $C$ — the brittle ridge is gone.
- **A Gibbs-style centre.** The regularised problem organises itself around one operator, the matrix-exponential Gibbs kernel $K = \exp(-C/\varepsilon)$ — the next section builds it.
- **Well-conditioned solves.** The strictly convex objective is friendlier to interior-point methods, the same practical payoff entropy bought in the classical case.

This is the entry in the dictionary directly across from `03/09`. There the entropy was Shannon's and the variable was a non-negative matrix; here the entropy is von Neumann's and the variable is a positive-semidefinite operator. We solve the problem with cvxpy's `von_neumann_entr` atom, which lets us write the entropic SDP as the `04/05` SDP plus a single term. The explicit iteration that *reaches* this optimum — the operator analogue of Sinkhorn's alternating rescaling — is `04/07`.

## The matrix-exponential Gibbs kernel

In `03/09` the Gibbs kernel was the **entrywise** exponential of the cost, $K_{ij} = e^{-C_{ij}/\varepsilon}$ — one scalar exponential per matrix entry. When the cost is an operator rather than a table of numbers, the faithful lift replaces the entrywise exponential by the **matrix exponential**:

$$ K \,=\, \exp\!\big(-C/\varepsilon\big), $$

defined through the eigendecomposition of $C$: diagonalise $C = U\,\Lambda\,U^\dagger$ and exponentiate the eigenvalues, $K = U\,e^{-\Lambda/\varepsilon}\,U^\dagger$. Two structural facts fall out immediately, and a third is the genuinely new quantum content:

- $K$ **shares the eigenvectors of $C$** and has eigenvalues $e^{-\lambda/\varepsilon}$ — strictly positive, so $K$ is **Hermitian and positive definite**, a legitimate (unnormalised) operator weight.
- For a **diagonal** cost (the commuting case) the matrix exponential acts entrywise on the diagonal, so $K_{ii} = e^{-C_{ii}/\varepsilon}$ and $K$ collapses **exactly** to the entrywise classical kernel of `03/09`.
- For a **non-diagonal** cost the matrix exponential mixes the eigenbasis of $C$ into off-diagonal entries that no entrywise rule could produce — this operator mixing is what the quantum lift adds.

We build it on the same quadratic position cost that carried `04/05`, with positions $\{0, 1\}$, at $\varepsilon = 0.5$, using `matrix_gibbs_kernel`.

In [ ]:
# Same cost operator the module has used since 04/05: C = (X_A (x) I - I (x) X_B)^2.
C = quadratic_position_cost([0.0, 1.0])

epsilon = 0.5
K = matrix_gibbs_kernel(C, epsilon)

# Structure checks: Hermitian, positive definite, and same eigenvectors as C.
is_hermitian = np.allclose(K, K.conj().T)
min_eig_K = float(np.min(np.linalg.eigvalsh(0.5 * (K + K.conj().T))))
eig_C = np.linalg.eigvalsh(0.5 * (C + C.conj().T))
eig_K = np.linalg.eigvalsh(0.5 * (K + K.conj().T))

print("cost operator C (quadratic position, positions {0, 1}):")
print(np.round(C.real, 4))
print()
print(f"matrix-exponential Gibbs kernel  K = expm(-C / {epsilon}):")
print(np.round(K.real, 4))
print()
print(f"K Hermitian?                 {is_hermitian}")
print(f"K positive definite?         {min_eig_K > 0}   (smallest eigenvalue {min_eig_K:.4f})")
print()
# Eigenvalues of K must equal exp(-eigenvalues(C) / eps): K and C share eigenvectors.
print(f"eigenvalues of C:                  {np.round(eig_C, 4)}")
print(f"exp(-eig(C) / eps):                {np.round(np.exp(-eig_C / epsilon), 4)}")
print(f"eigenvalues of K:                  {np.round(eig_K, 4)}")
# Sort both: eigvalsh returns eigenvalues ascending and exp(-x) is decreasing, so the
# two spectra match as sets even though their orders are reversed.
print(f"K eigenvalues == exp(-eig(C)/eps)? {np.allclose(np.sort(eig_K), np.sort(np.exp(-eig_C / epsilon)))}")
print()
# Diagonal cost => K collapses to the entrywise classical kernel of 03/09.
entrywise = np.exp(-np.diag(C).real / epsilon)
print(f"entrywise e^(-C_ii / eps):   {np.round(entrywise, 4)}")
print(f"diagonal of K:               {np.round(np.diag(K).real, 4)}")
print(f"max |off-diagonal| of K:     {np.max(np.abs(K - np.diag(np.diag(K)))):.2e}")

**Read the output.** The kernel $K$ is Hermitian with a strictly positive smallest eigenvalue, so it is positive definite, as promised. Its eigenvalues are exactly $e^{-\lambda/\varepsilon}$ applied to the eigenvalues $\lambda$ of $C$ — confirming that $K$ and $C$ share an eigenbasis and that the matrix exponential acts only on the spectrum. Because this particular $C$ is diagonal, that eigenbasis is the computational basis: $K$ is itself diagonal (its largest off-diagonal entry is zero to machine precision), and its diagonal $\{1,\ e^{-2},\ e^{-2},\ 1\}$ is exactly the entrywise kernel $e^{-C_{ii}/\varepsilon}$ of `03/09`. The two cells costing zero ($C_{ii}=0$) map to weight $1$; the two costing one map to $e^{-1/0.5} = e^{-2} \approx 0.1353$. On a commuting cost the quantum kernel *is* the classical kernel. The new operator content — off-diagonal mixing — switches on only when $C$ has off-diagonal structure of its own.

## Solving the entropic SDP

The implementation is the coupling SDP of `04/05` with one extra term in the objective — `- epsilon * cp.von_neumann_entr(plan)` — which is what `quantum_sinkhorn_sdp` assembles for us. It returns the optimal **objective value**, $\mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})$, together with the optimal plan.

To *see* the regulariser do its work, we want an example whose sharp plan is genuinely peaked, so that smoothing has something to smooth. The headline $|+\rangle\langle+|$ versus $I/2$ pair is a poignant special case where it would not: a pure marginal ($S=0$) against the maximally mixed one ($S=1$ bit) forces the joint entropy to sit at exactly $1$ bit for *every* coupling (the Araki–Lieb bound squeezes $|S_A - S_B| \le S_{AB} \le S_A + S_B$ shut), so the entropy bonus is constant over the feasible set and cannot move the plan — entropic and sharp optima coincide there. So we keep the same **quadratic position cost** but take a commuting pair with room to spread: two diagonal "position" distributions, $\rho_A = \mathrm{diag}(0.5, 0.3, 0.2)$ and $\rho_B = \mathrm{diag}(0.2, 0.3, 0.5)$ on positions $\{0, 1, 2\}$. We solve the unregularised SDP for the sharp reference, then the entropic SDP at $\varepsilon = 1$, and compare.

In [ ]:
# Three positions, two commuting diagonal states with room to spread.
C3 = quadratic_position_cost([0.0, 1.0, 2.0])
rho_a = np.diag([0.5, 0.3, 0.2]).astype(complex)
rho_b = np.diag([0.2, 0.3, 0.5]).astype(complex)

# Sharp reference: the unregularised coupling SDP of 04/05.
sharp_value, sharp_plan = quantum_ot_sdp(rho_a, rho_b, C3)

# Entropic solve: tr(C P) minus eps times S(P), in one extra term.
epsilon = 1.0
objective, plan_eps = quantum_sinkhorn_sdp(rho_a, rho_b, C3, epsilon=epsilon)


def von_neumann_entropy_nats(rho):
    """von Neumann entropy -tr(rho log rho) in nats (natural-log convention)."""
    vals = np.linalg.eigvalsh(0.5 * (rho + rho.conj().T))
    vals = vals[vals > 1e-12]
    return float(-np.sum(vals * np.log(vals)))


# Decompose the entropic objective into its two named pieces.
transport = float(np.real(np.trace(C3 @ plan_eps)))
S_eps = von_neumann_entropy_nats(plan_eps)
S_sharp = von_neumann_entropy_nats(sharp_plan)

print(f"entropic SDP objective  tr(C P) - eps*S(P)  = {objective:.4f}")
print(f"  transport part  tr(C * P_eps)             = {transport:.4f}")
print(f"  entropy part    S(P_eps)                  = {S_eps:.4f} nats  ({S_eps / np.log(2):.4f} bits)")
print(f"  eps * S(P_eps)                            = {epsilon * S_eps:.4f}")
print(f"  check: transport - eps*S                  = {transport - epsilon * S_eps:.4f}   (matches objective)")
print()
print(f"sharp plan (04/05 SDP):  transport = {sharp_value:.4f},  S = {S_sharp / np.log(2):.4f} bits")
print(f"entropic plan (eps={epsilon}):     S = {S_eps / np.log(2):.4f} bits   <- higher entropy => more spread")
print()
# An honest coupling: partial traces must return the prescribed marginals (to solver tolerance).
tr_b = partial_trace(plan_eps, keep=[0], dims=[3, 3])
tr_a = partial_trace(plan_eps, keep=[1], dims=[3, 3])
print(f"marginal A recovered (diag, want 0.5, 0.3, 0.2):  {np.round(np.diag(tr_b).real, 4)}")
print(f"marginal B recovered (diag, want 0.2, 0.3, 0.5):  {np.round(np.diag(tr_a).real, 4)}")

**Read the output.** The entropic SDP reports an **objective**, not a transport cost: it is $\mathrm{tr}(C\,\rho_{AB})$ *minus* $\varepsilon$ times the entropy, and the printed decomposition confirms the two pieces add back to that value. The story to read is in the last lines. The sharp plan of `04/05` carries about $1.97$ bits of entropy; the entropic plan at $\varepsilon = 1$ carries about $2.58$ bits. The bonus has pulled the coupling toward **higher entropy — more spread** — exactly as the dial promised. (Its transport cost is correspondingly no longer the minimum: at $\varepsilon = 1$ the optimum is happy to pay differently in cost to buy that extra spread, which is the regulariser working as designed.) And the plan is still an honest coupling — tracing out either subsystem returns the prescribed diagonal marginal to solver tolerance. We look at the two plans side by side next.

In [ ]:
# Show the sharp and the entropic plans as density-matrix heatmaps (real + imaginary parts).
# Couplings on a 3-position (x) 3-position space -> label each cell by its position pair |ij>.
position_pair_basis = [f"|{i}{j}>" for i in range(3) for j in range(3)]
viz.plot_density_matrix(
    sharp_plan,
    title=r"Sharp coupling $\rho^\star_{AB}$ -- unregularised SDP (04/05)",
    basis_labels=position_pair_basis,
)
plt.show()

viz.plot_density_matrix(
    plan_eps,
    title=rf"Entropic coupling $\rho^\star_{{AB,\,\varepsilon}}$ at $\varepsilon = {epsilon}$",
    basis_labels=position_pair_basis,
)
plt.show()

**Read the figures.** Each figure shows one coupling as two heatmaps — its real part on the left, its imaginary part on the right. Both imaginary parts are flat at zero: these commuting diagonal states have no coherence, so the optimal couplings are real, exactly as in module 03. The real parts tell the story. The **sharp** coupling (first figure) is concentrated: its weight sits on a short, near-permutation set of entries along the diagonal — the cheapest routes under the squared-distance cost, and little else. The **entropic** coupling (second figure) keeps that backbone but lights up additional entries around it: the same total mass, redistributed onto more cells, paying a little extra transport cost in exchange for the entropy bonus. This is the operator-level echo of the kernel picture from `03/09`, where a smaller $\varepsilon$ gave a thin bright diagonal and a larger $\varepsilon$ let it broaden. Here, at a single $\varepsilon$, we see the one move that broadening is built from: the sharp plan softened into a smoother one. How that softening varies as you turn the $\varepsilon$ dial — and the explicit iteration that produces it — is the next notebook.

## Your turn

**Warm-up.** Read the transport cost off the entropic optimum at two regularisation strengths using `quantum_sinkhorn_cost(rho_a, rho_b, C3, epsilon)`, which returns $\mathrm{tr}(C\,\rho^\star_{AB,\varepsilon})$ alone (the cost piece, without the entropy term). Compute it at $\varepsilon = 0.5$ and at $\varepsilon = 2$, and compare the two numbers to the sharp SDP value $\mathrm{tr}(C\,\rho^\star_{AB}) = 0.6$ from this section. Which $\varepsilon$ sits closer to the sharp cost, and does that match your sense of which one regularises more gently?

**Go further.** Confirm the entropic plan is a faithful coupling. Take the plan `plan_eps` you solved for above, trace out subsystem $B$ with `partial_trace(plan_eps, keep=[0], dims=[3, 3])` and subsystem $A$ with `keep=[1]`, and check that the two reduced states match $\rho_A$ and $\rho_B$ to a tolerance such as $10^{-6}$ (the interior-point residual). Why is it reassuring that *adding* the entropy bonus did not break the marginal constraints?

**Challenge.** Predict the two limits before computing anything, then test one. Reason from the objective $\mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})$: as $\varepsilon \to 0$ the entropy term vanishes and the problem returns to the sharp SDP of `04/05`; as $\varepsilon \to \infty$ the entropy term dominates and the optimum becomes the **highest-entropy** coupling with the right marginals, which is the product state $\rho_A \otimes \rho_B$ (build it with `tensor(rho_a, rho_b)`). Solve the entropic SDP at a large $\varepsilon$ (say $\varepsilon = 50$) and compare its plan to `tensor(rho_a, rho_b)` — for instance with `np.linalg.norm` of the difference. How close does it land, and why does that large-$\varepsilon$ endpoint make sense as "transport cost barely matters, spread as much as the marginals allow"?

## What you built

- You wrote the **entropic quantum OT** problem — the coupling SDP of `04/05` plus a von Neumann entropy bonus $-\varepsilon S(\rho_{AB})$ — and saw why that bonus makes the objective strongly convex, with a single optimum that moves smoothly with the data.
- You built the **matrix-exponential Gibbs kernel** $K = \exp(-C/\varepsilon)$, confirmed it is Hermitian positive definite with the eigenvectors of $C$ and eigenvalues $e^{-\lambda/\varepsilon}$, and watched it collapse to the entrywise classical kernel of `03/09` on a commuting cost.
- You solved the **entropic SDP** in cvxpy with one extra term, read its objective as transport cost minus $\varepsilon$ times entropy, and saw the entropic coupling come out **smoother** — higher entropy, more spread — than the sharp `04/05` plan, while still recovering both marginals.

Take a moment with the symmetry of it: the move that unlocked fast, stable optimal transport in module 03 lifted to operators almost word for word — Shannon became von Neumann, the entrywise kernel became a matrix exponential, and the regularised problem stayed convex and well-posed. You now hold the entropic problem and the kernel at its centre.

## What's next

In `07_operator_sinkhorn_iteration` we stop handing the whole problem to a black-box SDP solver and write the explicit **operator Sinkhorn iteration** — the matrix analogue of Sinkhorn's alternating row/column rescaling — that walks to this same entropic optimum by repeatedly rescaling the Gibbs kernel onto the marginals, and there we turn the $\varepsilon$ dial to watch the sharpness-versus-spread trade-off in full.

## References

- M. Cuturi, "Sinkhorn distances: lightspeed computation of optimal transport", *Advances in Neural Information Processing Systems* **26** (2013). DOI:10.48550/arXiv.1306.0895
- G. Peyré, L. Chizat, F.-X. Vialard & B. Schmitzer, "Quantum entropic regularization of matrix-valued optimal transport", *European Journal of Applied Mathematics* **30**, 1079–1102 (2019). DOI:10.1017/S0956792519000299
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091

**Previous:** `notebooks/04_QuantumOptimalTransport/05_solving_qot_in_cvxpy.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/07_operator_sinkhorn_iteration.ipynb`